In [ ]:
import tensorflow as tf
from tensorflow import keras
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Flatten
from keras.layers import Conv2D
from keras.layers import MaxPooling2D, Dropout
from keras import Model

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
import json
import cv2
import matplotlib.pyplot as plt
import random

In [ ]:
# f = pd.read_csv('./smash_dataframes/smash_master_dataframe.csv')

# l = []
# inalready = []
# while len(l) < 2000:
#     rand = random.randint(0, len(f))
#     if rand not in inalready:
#         if f.iloc[rand]['effect1'] == 'none' and f.iloc[rand]['effect2'] == 'none':
#             l.append(f.iloc[rand])
            
# temp = pd.DataFrame(l, columns=f.columns)

# df = pd.DataFrame(columns=df1.columns)

# df['filename'] = temp['files']
# df['effect'] = temp['effect1']

# df.to_csv('./effects/none_data.csv')

In [ ]:
df1 = pd.read_csv('./effects/ylink,ganon,effects,2.csv')
df2 = pd.read_csv('./effects/anle_effects.csv')
none = pd.read_csv('./effects/none_data.csv')

In [ ]:
df3 = pd.DataFrame(columns=df2.columns)
df3 = df3[['filename', 'effect']]
df3

In [ ]:
df3['filename'] = none['filename']
df3['effect'] = none['effect']
df3 = df3[0:200]

In [ ]:
df1 = df1[['filename', 'effect']]
df1

In [ ]:
df2 = df2[['filename','effect']]
df2

In [ ]:
master_df = pd.concat([df1, df2, df3], axis=0, ignore_index=True)
master_df

In [ ]:
le = LabelEncoder()
oh = OneHotEncoder()

output = le.fit_transform(master_df['effect'])
output2 = oh.fit_transform(output.reshape(-1, 1)).toarray()

In [ ]:
import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
    f.close()
with open("onehot_encoder.pkl", "wb") as f:
    pickle.dump(oh, f)
    f.close()

In [ ]:
output

In [ ]:
# data pipline
BATCH_SIZE = 32
data_directory = "./effects/"

# array of image files to read in parse_image
files = master_df['filename'].tolist()

files = tf.constant(files)

effect = tf.constant(output2)

# dataset
dataset = tf.data.Dataset.from_tensor_slices((files,effect))
dataset = dataset.shuffle(300)
ds_train = dataset.take(int(len(dataset)*.8)) # take first 80%
ds_validation = dataset.skip(int(len(dataset)*.8)).take(int(len(dataset)*.1)) # skip first 80%, take next 10%
ds_test = dataset.skip(int(len(dataset)*.9)) # skip first 90%, take rest

# image manipulation
# already normalizes pixel values
def parse_image(filename, label):
    image_string = tf.io.read_file(data_directory+filename)
    image_decoded = tf.io.decode_jpeg(image_string, channels=3)
    image = tf.image.convert_image_dtype(image_decoded, tf.float32)
    image = tf.image.resize(image, [224, 224])
    return image, label

# optimize pipeline
ds_train = ds_train.map(parse_image)
ds_train = ds_train.batch(BATCH_SIZE)
ds_train = ds_train.prefetch(tf.data.AUTOTUNE)
ds_test = ds_test.map(parse_image)
ds_test = ds_test.batch(BATCH_SIZE)
ds_test = ds_test.prefetch(tf.data.AUTOTUNE)
ds_validation = ds_validation.map(parse_image)
ds_validation = ds_validation.batch(BATCH_SIZE)
ds_validation = ds_validation.prefetch(tf.data.AUTOTUNE)

# batch iterator
iterator_train = iter(ds_train)
images_train, labels_train = iterator_train.get_next()
iterator_test = iter(ds_test)
images_test, labels_test = iterator_test.get_next()
iterator_validation = iter(ds_validation)
images_validation, labels_validation = iterator_validation.get_next()

In [ ]:
input_layer = Input((224, 224, 3))
conv2d1 = Conv2D(8, (3,3), activation='relu')(input_layer)
conv2d2 = Conv2D(8, (3,3), activation='relu')(conv2d1)
conv2d3 = Conv2D(8, (3,3), activation='relu')(conv2d2)
maxpool1 = MaxPooling2D((3,3))(conv2d3)
conv2d4 = Conv2D(16, (3,3), activation='relu')(maxpool1)
conv2d5 = Conv2D(16, (3,3), activation='relu')(conv2d4)
conv2d6 = Conv2D(16, (3,3), activation='relu')(conv2d5)
maxpool2 = MaxPooling2D((3,3))(conv2d6)
conv2d7 = Conv2D(32, (3,3), activation='relu')(maxpool2)
conv2d8 = Conv2D(32, (3,3), activation='relu')(conv2d7)
conv2d9 = Conv2D(32, (3,3), activation='relu')(conv2d8)
maxpool3 = MaxPooling2D((3,3))(conv2d9)
flatten = Flatten()(maxpool3)
dense1 = Dense(30, activation='relu')(flatten)
dense2 = Dense(3, activation='softmax')(dense1)

model = Model(input_layer, dense2)

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])
model.summary()

In [ ]:
# callback = tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=3)
# model.fit(ds_train, epochs=50, validation_data=(ds_validation), batch_size=16, callbacks=[callback])

history = model.fit(ds_train, epochs=50, validation_data=(ds_validation), batch_size=16)

In [ ]:
model.evaluate(ds_test)

In [ ]:
num_pictures = 50

for i in range(num_pictures):
    img_no = random.randint(0, len(master_df))
    img = cv2.imread('./effects/' + master_df['filename'][img_no])
    img2 = cv2.resize(img, (224, 224))
    img2 = img2.reshape(-1, 224, 224, 3)
    y_true = le.inverse_transform(oh.inverse_transform(np.array(output2[img_no]).reshape(1, -1)))
    y_pred = model.predict(img2)
    print('Path', './effects/' + master_df['filename'][img_no])
    print('True', y_true)
    print('Predicted', le.inverse_transform(oh.inverse_transform(y_pred)))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.show()

In [ ]:
# model.save('effects_model')

In [ ]:
# summarize history for accuracy
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for loss
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for precision
plt.plot(history.history['precision'])
plt.plot(history.history['val_precision'])
plt.title('model precision')
plt.ylabel('precision')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()
# summarize history for recall
plt.plot(history.history['recall'])
plt.plot(history.history['val_recall'])
plt.title('model recall')
plt.ylabel('recall')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

In [ ]:
# df_columns = ['x1', 'y1', 'x2', 'y2', 'width1', 'height1', 'width2', 'height2']
# diff = ['files', 'char1', 'move1', 'effect1', 'char2', 'move2', 'effect2']
# for i in diff:
#     df_columns.append(i)

In [ ]:
# ct = ColumnTransformer([
#         ('coord_transformer', StandardScaler(), ['x1', 'y1', 'x2', 'y2', 'width1', 'height1', 'width2', 'height2'])
#     ], remainder='passthrough')
# df = ct.fit_transform(df)
# df = pd.DataFrame(df, columns=df_columns)
# df